# Agricultural Water Demand and Irrigation Scheduling

This notebook walks through a small FAO-56 workflow for maize: it fetches daily weather inputs when Open-Meteo is available, falls back to a bundled sample CSV when it is not, computes reference ET$_0$, estimates crop water demand, and builds a simple irrigation schedule with a soil-water balance. The goal is to show the whole path from weather data to actionable irrigation numbers in a compact, reproducible tutorial.

## Setup

Install the optional extras used in this notebook:

```bash
pip install "aquascope[viz]"
# or, for the minimal package:
pip install aquascope
```

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if not (repo_root / "aquascope").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aquascope.agri import (
    crop_water_requirement,
    fetch_openmeteo_plan_inputs,
    get_kc,
    hargreaves,
    irrigation_schedule,
    penman_monteith_daily,
    penman_monteith_series,
    plan_irrigation,
)
from aquascope.agri.planner import default_season_end_date
from aquascope.agri.water_balance import SoilProperties

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)

## Fetch weather inputs

We request ET₀ and rainfall through the last day of the maize growing season. If the service cannot supply both for every day, we use the bundled weather record through its last available date. The calculations, plots and totals below cover only the reported weather window.

In [ ]:
latitude = 35.0
longitude = 139.0
elevation = 120.0
planting_date = date(2026, 4, 1)
season_end = default_season_end_date("maize", planting_date)

data_path = repo_root / "notebooks" / "data" / "agri_weather_sample.csv"
sample_weather_df = pd.read_csv(data_path, parse_dates=["date"])
sample_weather_df = sample_weather_df.set_index("date").sort_index()

weather_cols = ["t_min", "t_max", "rh_min", "rh_max", "wind_speed", "solar_radiation"]
pm_weather_df = sample_weather_df[weather_cols]

sample_pm_eto = penman_monteith_series(pm_weather_df, latitude=latitude, elevation=elevation)
sample_precip = sample_weather_df["precipitation_sum"]

def daily_weather(eto, precip, end):
    days = pd.date_range(planting_date, end, freq="D")
    weather = pd.DataFrame({"eto_mm": eto, "precip_mm": precip}, index=days)
    if weather.empty or not np.isfinite(weather.to_numpy()).all() or (weather < 0).any().any():
        raise ValueError("Each planning day needs a valid ET₀ and precipitation value.")
    return weather

try:
    fetched_eto, fetched_precip = fetch_openmeteo_plan_inputs(
        latitude, longitude, planting_date.isoformat(), season_end.isoformat()
    )
    weather = daily_weather(fetched_eto, fetched_precip, season_end)
    plan_source = "Open-Meteo"
except Exception as exc:
    sample_end = min(season_end, sample_weather_df.index.max().date())
    weather = daily_weather(sample_pm_eto, sample_precip, sample_end)
    plan_source = f"sample CSV fallback ({exc.__class__.__name__})"

plan_eto_series = weather["eto_mm"]
plan_precip_series = weather["precip_mm"]

def covered_days(table):
    # The crop functions return all growth stages, even for a shorter forcing record.
    return table.loc[pd.to_datetime(table["date"]).isin(weather.index)].copy()

display(weather.head())
print(f"{plan_source}: {weather.index[0]:%Y-%m-%d} to {weather.index[-1]:%Y-%m-%d} "
      f"({len(weather)} of {(season_end - planting_date).days + 1} growing-season days)")

## Reference ET$_0$

The bundled weather file lets us demonstrate the daily FAO-56 calculation directly. The first call uses `penman_monteith_series` on a weather DataFrame, and the second shows a one-day `penman_monteith_daily` example. A simple `hargreaves` call is included as a temperature-only fallback.

In [ ]:
daily_row = sample_weather_df.iloc[0]
daily_doy = daily_row.name.timetuple().tm_yday

eto_one_day = penman_monteith_daily(
    t_min=float(daily_row["t_min"]),
    t_max=float(daily_row["t_max"]),
    rh_min=float(daily_row["rh_min"]),
    rh_max=float(daily_row["rh_max"]),
    u2=float(daily_row["wind_speed"]),
    rs=float(daily_row["solar_radiation"]),
    latitude=latitude,
    elevation=elevation,
    doy=daily_doy,
)

ra = 24.0  # simple example value for the Hargreaves demonstration
eto_hargreaves = hargreaves(float(daily_row["t_min"]), float(daily_row["t_max"]), ra)

print("First five ET0 values from the sample weather DataFrame:")
display(sample_pm_eto.head().to_frame(name="eto_mm"))
print(f"Single-day Penman-Monteith ET0: {eto_one_day:.2f} mm/day")
print(f"Hargreaves example ET0: {eto_hargreaves:.2f} mm/day")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
sample_pm_eto.plot(ax=ax, color="#0f766e", lw=2)
ax.set_title("Reference ET$_0$ from the sample weather file")
ax.set_ylabel("ET$_0$ (mm/day)")
ax.set_xlabel("Date")
plt.tight_layout()

## Crop coefficients

For this tutorial we use maize. The single crop coefficient starts low during establishment, rises through development, reaches its mid-season plateau, and tapers during late season.

In [ ]:
kc = get_kc("maize")
display(pd.Series(kc, name="Kc"))
print("Initial = establishment, mid = peak canopy demand, late = senescence / harvest decline.")

## Crop water requirement

ET₀ and the maize crop coefficient give the daily crop evapotranspiration (ETc). The total covers the days with weather inputs, which may end before the crop reaches harvest.

In [ ]:
cwr_df = covered_days(crop_water_requirement(
    plan_eto_series,
    crop="maize",
    planting_date=planting_date,
))

display(cwr_df[["date", "stage", "kc", "eto", "etc"]].head())
print(f"Crop water requirement (ETc) over the weather window: {cwr_df['etc'].sum():.2f} mm")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(cwr_df["date"], cwr_df["etc"], color="#b45309", lw=2)
ax.set_title("Maize crop water requirement")
ax.set_ylabel("ETc (mm/day)")
ax.set_xlabel("Date")
plt.tight_layout()

## Irrigation scheduling

The daily requirement compares crop demand with effective rainfall. The soil-water plan also carries water between days in the root zone, initially at field capacity. Once the previous day's depletion reaches the readily available water threshold, the plan schedules a refill. A day can therefore have an irrigation requirement even when the model schedules no application.

In [ ]:
efficiency = 0.7
sched = covered_days(irrigation_schedule(
    plan_eto_series,
    plan_precip_series,
    crop="maize",
    planting_date=planting_date,
    efficiency=efficiency,
))

soil = SoilProperties(field_capacity=0.30, wilting_point=0.15, root_depth=1.0)
plan = plan_irrigation(
    crop="maize",
    planting_date=planting_date,
    eto_series=plan_eto_series,
    precip_series=plan_precip_series,
    soil=soil,
    efficiency=efficiency,
)

balance = covered_days(plan.balance)
applications = balance.loc[balance["irrigation_mm"] > 0, ["date", "irrigation_mm"]].copy()
applications = applications.rename(columns={"irrigation_mm": "gross_applied_mm"})
applications["net_applied_mm"] = efficiency * applications["gross_applied_mm"]

display(sched[["date", "etc", "effective_rain", "net_irrigation", "gross_irrigation"]].head())
display(applications.round(2))
print(f"Modeled applications: {len(applications)}")

fig, (demand_ax, applied_ax) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
demand_ax.plot(sched["date"], sched["net_irrigation"], label="Net requirement")
demand_ax.plot(sched["date"], sched["gross_irrigation"], linestyle="--", label="Gross requirement")
demand_ax.set_ylabel("Demand (mm/day)")
demand_ax.legend()
applied_ax.bar(applications["date"], applications["gross_applied_mm"], label="Gross pumped")
applied_ax.bar(applications["date"], applications["net_applied_mm"], label="Net to root zone")
applied_ax.set_ylabel("Application (mm)")
applied_ax.set_xlabel("Date")
applied_ax.legend()
plt.tight_layout()

## Summary

The daily water requirements and the modeled applications answer different planning questions. The first totals show how much it would take to replace each day's deficit immediately. The application totals show when the modeled soil actually needs refilling and the gross depth applied at 70% efficiency.

In [ ]:
summary = pd.DataFrame([{
    "weather_start": weather.index[0].date(),
    "weather_end": weather.index[-1].date(),
    "days_with_weather": len(weather),
    "total_etc_mm": sched["etc"].sum(),
    "effective_rain_mm": sched["effective_rain"].sum(),
    "daily_net_requirement_mm": sched["net_irrigation"].sum(),
    "daily_gross_requirement_mm": sched["gross_irrigation"].sum(),
    "application_days": len(applications),
    "gross_applied_mm": applications["gross_applied_mm"].sum(),
    "net_applied_mm": applications["net_applied_mm"].sum(),
}])
display(summary.round(2))

## Error handling note

When a user asks for a crop that is not in the FAO-56 lookup table, AquaScope raises a clear error. Catching that error lets you show a friendlier message in notebooks or apps.

In [ ]:
try:
    get_kc("invalid_crop")
except ValueError as exc:
    print(f"Could not look up crop coefficients: {exc}")